# HMM
- hidden state: offset
- TVT = 수직 깊이 + offset


In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, glob
from pathlib import Path

In [22]:
DATA_DIR = Path("../data/raw")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

_train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv')))
TRAIN_WIDS = [os.path.basename(f).split('__')[0] for f in _train_files]
_test_files = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WIDS = [os.path.basename(f).split('__')[0] for f in _test_files]

print(f"훈련 데이터 총 우물 수: {len(TRAIN_WIDS)}")
print(f"훈련 데이터 우물 ID: {TRAIN_WIDS[:5]}")

def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw

훈련 데이터 총 우물 수: 773
훈련 데이터 우물 ID: ['000d7d20', '00bbac68', '00e12e8b', '015fe0d2', '01869cd4']


In [23]:
import numpy as np
from scipy.special import logsumexp

def hmm_grid_filter(well_md, well_tvd, well_gr, tw_tvt, tw_gr,
                    offset_min=-50.0, offset_max=50.0, step=0.25,
                    sigma=10.0, dip_sigma=0.5):
    """
    Input
    - well_md   : hw의 심도
    - well_tvd  : 각 MD에서 수직 심도
    - well_gr   : 각 MD에서 GR 값
    - tw_tvt    : tw의 수직위치(TVT) 격자
    - tw_gr     : tw의 GR 값
    """
    # 1) 오프셋 격자: offset 후보 범위를 격자로 생성
    offsets = np.arange(offset_min, offset_max + step, step)    # (S, )
    n_states = len(offsets)     # state 개수 S
    n_obs = len(well_md)        # 깊이 포인트 개수 N

    # 2) 방출 로그 확률 emit[t, s] = 깊이 t에서 offset이  offsets[s]일 때 GR이 맞을 로그가능도
    # pred_tvt = TVD + offsets[s]
    pred_tvt = well_tvd[:, None] + offsets[None, :]             # (N, S)
    # pred_tvt에서 예상 tw GR을 보간으로 생성
    pred_gr = np.interp(pred_tvt.ravel(), tw_tvt, tw_gr).reshape(n_obs, n_states)
    # 실제 GR과의 오차로 가우시안 로그가능도 계산
    emit = -0.5 * ((well_gr[:, None] - pred_gr) / sigma) ** 2   # (N, S)

    # 3) 전이 로그 확률 log_trans[i, j] = offsets[i]에서 offsets[j]로 한 스텝 변할 로그확률
    diff = offsets[:, None] - offsets[None, :]                  # (S, S)
    log_trans = -0.5 * (diff/dip_sigma) ** 2                    # 멀수록 낮은 확률

    # 4) Forward: 앞에서 뒤로 누적 log_alpha[t, s]
    log_alpha = np.empty((n_obs, n_states))
    log_alpha[0] = emit[0]
    log_alpha[0] -= logsumexp(log_alpha[0])                     # 정규화
    for t in range(1, n_obs):
        prev = log_alpha[t-1][:, None] + log_trans              # (S, S)
        log_alpha[t] = logsumexp(prev, axis=0) + emit[t]        # 경로 합치고 방출 더함
        log_alpha[t] -= logsumexp(log_alpha[t])

    # 5) Backward: 뒤에서 앞으로 누적 log_beta[t, s]
    log_beta = np.empty((n_obs, n_states))
    log_beta[-1] = 0.0
    for t in range(n_obs-2, -1, -1):
        nxt = log_trans + emit[t+1][None, :] + log_beta[t+1][None, :]   # (S, S)
        log_beta[t] = logsumexp(nxt, axis=1)
        log_beta[t] -= logsumexp(log_beta[t])

    # 6) 사후분포 posterior[t, s] = 모든 정보를 합친 각 깊이의 오프셋 확률분포
    log_post = log_alpha + log_beta
    log_post -= logsumexp(log_post, axis=1, keepdims=True)
    post = np.exp(log_post)

    # 7) 예측
    offset_hat = post @ offsets
    tvt_hat = well_tvd + offset_hat

    return tvt_hat, post, offsets

In [ ]:
import pandas as pd

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


N_EVAL = 20
rows = []
for wid in TRAIN_WIDS[:N_EVAL]:
    hw, tw = load_well(wid, 'train')
    mask = hw['TVT_input'].isna().values
    if mask.sum() == 0:
        continue
    y_true = hw['TVT'].values.astype(float)
    kn = hw[hw['TVT_input'].notna()]

    # ★ TVT = −Z + C  →  well_tvd = −Z + base 
    base = float(np.median(kn['TVT_input'].values + kn['Z'].values))   # median(TVT + Z)
    well_md  = hw['MD'].values.astype(float)
    well_tvd = -hw['Z'].values.astype(float) + base
    well_gr  = hw['GR'].interpolate(limit_direction='both').fillna(hw['GR'].mean()).values.astype(float)

    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    tvt_hat, _, _ = hmm_grid_filter(well_md, well_tvd, well_gr, tw_tvt, tw_gr, 
                                    offset_min=-300, offset_max=300, step=1.0)
    rows.append({'wid': wid, 'rmse': rmse(y_true[mask], tvt_hat[mask])})
    print(f'{wid}  HMM RMSE = {rows[-1]["rmse"]:6.3f} ft')

df_hmm = pd.DataFrame(rows)
print(f'\n평균 HMM RMSE = {df_hmm["rmse"].mean():.3f} ft  ({len(df_hmm)} wells)')


000d7d20  HMM RMSE = 72.798 ft


In [ ]:
import pandas as pd

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

hw, tw = load_well('000d7d20', 'train')
mask = hw['TVT_input'].isna().values
y_true = hw['TVT'].values.astype(float)
kn = hw[hw['TVT_input'].notna()]
base = float(np.median(kn['TVT_input'].values + kn['Z'].values))
well_md  = hw['MD'].values.astype(float)
well_tvd = -hw['Z'].values.astype(float) + base
well_gr  = hw['GR'].interpolate(limit_direction='both').fillna(hw['GR'].mean()).values.astype(float)
tw_s = tw.sort_values('TVT'); tw_tvt = tw_s['TVT'].values.astype(float)
tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

tvt_hat, post, offsets = hmm_grid_filter(well_md, well_tvd, well_gr, tw_tvt, tw_gr,
                                         offset_min=-50, offset_max=50, step=0.25)
print('known 구간 RMSE :', rmse(y_true[~mask], tvt_hat[~mask]))
print('hidden 구간 RMSE:', rmse(y_true[mask], tvt_hat[mask]))
print('post 엔트로피(평균):', float(-(post*np.log(post+1e-12)).sum(1).mean()))


known 구간 RMSE : 18.382283500940147
hidden 구간 RMSE: 52.71326245539091
post 엔트로피(평균): 2.4958808632569527


In [ ]:
def hmm_viterbi(well_md, well_tvd, well_gr, tw_tvt, tw_gr, known_offset=None,
                offset_min=-50, offset_max=50, step=0.25,
                sigma=10.0, dip_sigma=2.0, anchor_sigma=0.3):
    offsets = np.arange(offset_min, offset_max + step, step)
    n_states, n_obs = len(offsets), len(well_md)

    # emission: GR 매칭
    pred_tvt = well_tvd[:, None] + offsets[None, :]
    pred_gr = np.interp(pred_tvt.ravel(), tw_tvt, tw_gr).reshape(n_obs, n_states)
    emit = -0.5 * ((well_gr[:, None] - pred_gr) / sigma) ** 2

    #  known anchor
    if known_offset is not None:
        for t in range(n_obs):
            if not np.isnan(known_offset[t]):
                emit[t] = -0.5 * ((offsets - known_offset[t]) / anchor_sigma) ** 2

    # transition
    diff = offsets[:, None] - offsets[None, :]
    log_trans = -0.5 * (diff / dip_sigma) ** 2

    # Viterbi
    delta = np.empty((n_obs, n_states)); psi = np.zeros((n_obs, n_states), dtype=int)
    delta[0] = emit[0]
    for t in range(1, n_obs):
        scores = delta[t-1][:, None] + log_trans      # (S_prev, S_cur)
        psi[t] = scores.argmax(0)
        delta[t] = scores.max(0) + emit[t]
    path = np.empty(n_obs, dtype=int)
    path[-1] = delta[-1].argmax()
    for t in range(n_obs-2, -1, -1):
        path[t] = psi[t+1][path[t+1]]

    return well_tvd + offsets[path], path, offsets


In [24]:
import pandas as pd

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

In [ ]:
N_EVAL = 20
rows = []
for wid in TRAIN_WIDS[:N_EVAL]:
    hw, tw = load_well(wid, 'train')
    mask = hw['TVT_input'].isna().values
    if mask.sum() == 0:
        continue
    y_true = hw['TVT'].values.astype(float)
    kn = hw[hw['TVT_input'].notna()]

    base = float(np.median(kn['TVT_input'].values + kn['Z'].values))
    well_md  = hw['MD'].values.astype(float)
    well_tvd = -hw['Z'].values.astype(float) + base
    well_gr  = hw['GR'].interpolate(limit_direction='both').fillna(hw['GR'].mean()).values.astype(float)

    known_offset = np.full(len(hw), np.nan)
    known_offset[~mask] = hw['TVT_input'].values[~mask] - well_tvd[~mask]

    tw_s = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    tvt_hat, _, _ = hmm_viterbi(well_md, well_tvd, well_gr, tw_tvt, tw_gr,
                                known_offset=known_offset)
    rows.append({'wid': wid, 'rmse': rmse(y_true[mask], tvt_hat[mask])})
    print(f'{wid}  Viterbi RMSE = {rows[-1]["rmse"]:6.3f} ft')

df = pd.DataFrame(rows)
print(f'\n평균 = {df["rmse"].mean():.3f} ft')


000d7d20  Viterbi RMSE = 58.020 ft
00bbac68  Viterbi RMSE = 121.160 ft
00e12e8b  Viterbi RMSE = 113.131 ft
015fe0d2  Viterbi RMSE = 75.572 ft
01869cd4  Viterbi RMSE = 120.420 ft
01982c1d  Viterbi RMSE = 73.418 ft
028d7b28  Viterbi RMSE = 187.958 ft
02e7fe5a  Viterbi RMSE = 100.365 ft
0390d174  Viterbi RMSE = 94.137 ft
03a935ae  Viterbi RMSE = 23.372 ft
044af7d1  Viterbi RMSE = 91.035 ft
0498acab  Viterbi RMSE = 96.698 ft
052d64df  Viterbi RMSE = 75.989 ft
05948241  Viterbi RMSE = 131.342 ft
059c8f24  Viterbi RMSE = 163.552 ft
05a0ee4d  Viterbi RMSE = 140.479 ft
060ab2b8  Viterbi RMSE = 115.338 ft
06df5958  Viterbi RMSE = 54.742 ft
071d7b45  Viterbi RMSE = 114.591 ft
0849b4e2  Viterbi RMSE = 110.994 ft

평균 = 103.116 ft


In [ ]:
from scipy.spatial import cKDTree

# 1) train 전체 점구름
def build_point_cloud(wids):
    XYZ, C, W = [], [], []
    for wid in wids:
        h, _ = load_well(wid, 'train')
        z = h['Z'].values.astype(float)
        XYZ.append(np.c_[h['X'].values, h['Y'].values, z])
        C.append(h['TVT'].values.astype(float) + z)       # C = TVT + Z
        W.append(np.full(len(h), wid))
    return np.vstack(XYZ), np.concatenate(C), np.concatenate(W)

XYZ_all, C_all, W_all = build_point_cloud(TRAIN_WIDS)
SCALE = XYZ_all.std(0)                                      # 축별 정규화
tree  = cKDTree(XYZ_all / SCALE)

# 2) 현재 우물 각 점의 C prior

def well_mean_xy(wid):
    hw, _ = load_well(wid, 'train')
    return float(hw['X'].mean()), float(hw['Y'].mean())

TRAIN_POS = np.array([well_mean_xy(w) for w in TRAIN_WIDS])   # shape (773, 2)
print('TRAIN_POS:', TRAIN_POS.shape)

def spatial_c_prior(hw, wid, K_wells=8, k=15):
    # 1) XY 거리로 이웃 우물 K개 (자기 제외)
    tx, ty = float(hw['X'].mean()), float(hw['Y'].mean())
    d = np.sqrt(((TRAIN_POS - [tx, ty]) ** 2).sum(1))
    order = np.argsort(d)
    neigh = [TRAIN_WIDS[i] for i in order if TRAIN_WIDS[i] != wid][:K_wells]

    # 2) 이웃 우물들의 점구름 (X,Y,Z, C=TVT+Z)
    XYZ, C = [], []
    for nb in neigh:
        h, _ = load_well(nb, 'train')
        z = h['Z'].values.astype(float)
        XYZ.append(np.c_[h['X'].values, h['Y'].values, z])
        C.append(h['TVT'].values.astype(float) + z)
    XYZ = np.vstack(XYZ) / SCALE
    C = np.concatenate(C)

    # 3) 현재 점에서 이웃 점 k개 평균 C
    tree = cKDTree(XYZ)
    q = np.c_[hw['X'].values, hw['Y'].values, hw['Z'].values].astype(float) / SCALE
    _, idx = tree.query(q, k=k)
    return C[idx].mean(axis=1)

# 검증
for wid in TRAIN_WIDS[:10]:
    hw, _ = load_well(wid, 'train')
    mask = hw['TVT_input'].isna().values
    z = hw['Z'].values.astype(float)

    c_prior = spatial_c_prior(hw, wid)              
    known_c = hw['TVT_input'].values + z
    off = np.nanmean((known_c - c_prior)[~mask])
    tvt_prior = (c_prior + off) - z                 # 보정된 C

    r = rmse(hw['TVT'].values[mask], tvt_prior[mask])
    base_c = np.nanmean(known_c[~mask])
    r_base = rmse(hw['TVT'].values[mask], (base_c - z)[mask])
    print(f'{wid}  보정 C-prior={r:6.2f}  |  known-C 외삽={r_base:6.2f} ft')



TRAIN_POS: (773, 2)
000d7d20  보정 C-prior= 11.85  |  known-C 외삽= 65.40 ft
00bbac68  보정 C-prior= 52.09  |  known-C 외삽=125.22 ft
00e12e8b  보정 C-prior=  6.51  |  known-C 외삽=102.47 ft
015fe0d2  보정 C-prior= 69.06  |  known-C 외삽= 96.75 ft
01869cd4  보정 C-prior= 12.39  |  known-C 외삽=149.84 ft
01982c1d  보정 C-prior= 89.78  |  known-C 외삽=108.44 ft
028d7b28  보정 C-prior= 72.34  |  known-C 외삽=197.62 ft
02e7fe5a  보정 C-prior= 12.47  |  known-C 외삽= 79.73 ft
0390d174  보정 C-prior= 27.79  |  known-C 외삽=126.71 ft
03a935ae  보정 C-prior= 41.62  |  known-C 외삽= 58.22 ft


In [ ]:
from scipy.special import logsumexp

def hmm_spatial(well_md, well_tvd, well_gr, tw_tvt, tw_gr,
                offset_spatial=None, sp_sigma=15.0, known_offset=None,
                offset_min=-50, offset_max=50, step=0.25,
                sigma=10.0, dip_sigma=2.0, anchor_sigma=0.3, use_viterbi=False):
    offsets = np.arange(offset_min, offset_max + step, step)
    n_states, n_obs = len(offsets), len(well_md)

    # 1) GR emission
    pred_tvt = well_tvd[:, None] + offsets[None, :]
    pred_gr  = np.interp(pred_tvt.ravel(), tw_tvt, tw_gr).reshape(n_obs, n_states)
    emit = -0.5 * ((well_gr[:, None] - pred_gr) / sigma) ** 2

    # 2) 공간 anchor: 각 점의 offset을 공간 prior 쪽으로
    if offset_spatial is not None:
        sp = np.asarray(sp_sigma, dtype=float)
        if sp.ndim == 0:
            sp = np.full(n_obs, float(sp))
        emit += -0.5 * ((offsets[None, :] - offset_spatial[:, None]) / sp[:, None]) ** 2

    # 3) known anchor: 정답 구간은 실제 offset에
    if known_offset is not None:
        for t in range(n_obs):
            if not np.isnan(known_offset[t]):
                emit[t] = -0.5 * ((offsets - known_offset[t]) / anchor_sigma) ** 2

    # 4) transition
    diff = offsets[:, None] - offsets[None, :]
    log_trans = -0.5 * (diff / dip_sigma) ** 2

    if use_viterbi:
        delta = np.empty((n_obs, n_states)); psi = np.zeros((n_obs, n_states), dtype=int)
        delta[0] = emit[0]
        for t in range(1, n_obs):
            sc = delta[t-1][:, None] + log_trans
            psi[t] = sc.argmax(0); delta[t] = sc.max(0) + emit[t]
        path = np.empty(n_obs, dtype=int); path[-1] = delta[-1].argmax()
        for t in range(n_obs-2, -1, -1):
            path[t] = psi[t+1][path[t+1]]
        return well_tvd + offsets[path]
    else:
        la = np.empty((n_obs, n_states)); la[0] = emit[0] - logsumexp(emit[0])
        for t in range(1, n_obs):
            la[t] = logsumexp(la[t-1][:, None] + log_trans, axis=0) + emit[t]
            la[t] -= logsumexp(la[t])
        lb = np.zeros((n_obs, n_states))
        for t in range(n_obs-2, -1, -1):
            lb[t] = logsumexp(log_trans + emit[t+1][None, :] + lb[t+1][None, :], axis=1)
            lb[t] -= logsumexp(lb[t])
        post = np.exp(la + lb - logsumexp(la + lb, axis=1, keepdims=True))
        return well_tvd + post @ offsets


In [ ]:
import pickle, time
N_WELLS       = 773
PF_SEEDS      = 8  

def spatial_c_prior_full(hw, wid, K_wells=8, k=15):
    tx, ty = float(hw['X'].mean()), float(hw['Y'].mean())
    order = np.argsort(np.sqrt(((TRAIN_POS - [tx, ty]) ** 2).sum(1)))
    neigh = [TRAIN_WIDS[i] for i in order if TRAIN_WIDS[i] != wid][:K_wells]

    XYZ, C = [], []
    for nb in neigh:
        h, _ = load_well(nb, 'train')
        z = h['Z'].values.astype(float)
        XYZ.append(np.c_[h['X'].values, h['Y'].values, z])
        C.append(h['TVT'].values.astype(float) + z)
    XYZ = np.vstack(XYZ) / SCALE
    C   = np.concatenate(C)

    tree = cKDTree(XYZ)
    q = np.c_[hw['X'].values, hw['Y'].values, hw['Z'].values].astype(float) / SCALE
    _, idx = tree.query(q, k=k)
    return C[idx].mean(axis=1), C[idx].std(axis=1)   # (c_prior, c_std)

In [ ]:
N_EVAL = 20
rows = []
for wid in TRAIN_WIDS[:N_EVAL]:
    hw, tw = load_well(wid, 'train')
    mask = hw['TVT_input'].isna().values
    if mask.sum() == 0:
        continue
    y_true = hw['TVT'].values.astype(float)
    z = hw['Z'].values.astype(float)
    kn = hw[hw['TVT_input'].notna()]

    base = float(np.median(kn['TVT_input'].values + kn['Z'].values))
    well_md  = hw['MD'].values.astype(float)
    well_tvd = -z + base
    well_gr  = hw['GR'].interpolate(limit_direction='both').fillna(hw['GR'].mean()).values.astype(float)
    tw_s = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    c_prior, c_std = spatial_c_prior_full(hw, wid)
    known_c = hw['TVT_input'].values + z
    off = np.nanmean((known_c - c_prior)[~mask])
    offset_spatial = (c_prior + off) - base
    sp_sigma = np.clip(c_std, 5.0, 60.0)

    known_offset = np.full(len(hw), np.nan)
    known_offset[~mask] = hw['TVT_input'].values[~mask] - well_tvd[~mask]

    tvt_hat = hmm_spatial(well_md, well_tvd, well_gr, tw_tvt, tw_gr,
                          offset_spatial=offset_spatial, sp_sigma=sp_sigma,
                          known_offset=known_offset, use_viterbi=False)
    rows.append({'wid': wid, 'rmse': rmse(y_true[mask], tvt_hat[mask])})
    print(f'{wid}  HMM+공간 RMSE = {rows[-1]["rmse"]:6.3f} ft')

df = pd.DataFrame(rows)
print(f'\n평균 = {df["rmse"].mean():.3f} ft')


000d7d20  HMM+공간 RMSE = 26.105 ft
00bbac68  HMM+공간 RMSE = 74.328 ft
00e12e8b  HMM+공간 RMSE = 59.247 ft
015fe0d2  HMM+공간 RMSE = 61.546 ft
01869cd4  HMM+공간 RMSE = 93.080 ft
01982c1d  HMM+공간 RMSE = 61.245 ft
028d7b28  HMM+공간 RMSE = 156.761 ft


: 